# Classification of Pet's Faces

Lab Assignment from [AI for Beginners Curriculum](https://github.com/microsoft/ai-for-beginners).

### Getting the Data

In this assignment, we will focus on relatively simple classification task - classification of pet's faces. We will use the [Oxford-IIIT Pet Dataset](https://www.robots.ox.ac.uk/~vgg/data/pets/), which contains images of 37 different breeds of dogs and cats. Let's start by downloading and visualizing the dataset.

**Note:** The Oxford-IIIT Pet Dataset contains full pet images. The images will be organized by breed in the extracted folder.

In [ ]:
!wget https://thor.robots.ox.ac.uk/~vgg/data/pets/images.tar.gz
!tar xfz images.tar.gz
!rm images.tar.gz

We will define generic function to display a series of images from a list:

In [ ]:
import matplotlib.pyplot as plt
import os
from PIL import Image
import numpy as np

def display_images(l,titles=None,fontsize=12):
    n=len(l)
    fig,ax = plt.subplots(1,n)
    for i,im in enumerate(l):
        ax[i].imshow(im)
        ax[i].axis('off')
        if titles is not None:
            ax[i].set_title(titles[i],fontsize=fontsize)
    fig.set_size_inches(fig.get_size_inches()*n)
    plt.tight_layout()
    plt.show()

Now let's traverse all class subdirectories and plot first few images of each class:

In [ ]:
# Note: The Oxford-IIIT Pet Dataset extracts to a folder named 'images'
# Images are named by breed (e.g., 'Abyssinian_1.jpg')
# We need to organize them into breed-specific subdirectories
import os
from collections import defaultdict

# Organize images by breed
if not os.path.exists('petfaces'):
    os.makedirs('petfaces')
    for img_file in os.listdir('images'):
        if img_file.endswith(('.jpg', '.png')):
            # Extract breed name from filename (everything before the last underscore and number)
            breed = '_'.join(img_file.split('_')[:-1])
            breed_dir = os.path.join('petfaces', breed)
            if not os.path.exists(breed_dir):
                os.makedirs(breed_dir)
            os.rename(os.path.join('images', img_file), os.path.join(breed_dir, img_file))

for cls in os.listdir('petfaces'):
    cls_path = os.path.join('petfaces', cls)
    # Skip files, only process directories
    if not os.path.isdir(cls_path):
        continue
    print(cls)
    display_images([Image.open(os.path.join(cls_path, x)) 
                    for x in os.listdir(cls_path)[:10]])

Let's also define the number of classes in our dataset:

In [ ]:
num_classes = len(os.listdir('petfaces'))
num_classes

## Preparing dataset for Deep Learning

To start training our neural network, we need to convert all images to tensors, and also create tensors corresponding to labels (class numbers). Most neural network frameworks contain simple tools for dealing with images:
* In Tensorflow, use `tf.keras.preprocessing.image_dataset_from_directory`
* In PyTorch, use `torchvision.datasets.ImageFolder`

As you have seen from the pictures above, all of them are close to square image ratio, so we need to resize all images to square size. Also, we can organize images in minibatches.

In [ ]:
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader, random_split

# Define transforms for the images
transform = transforms.Compose([
    transforms.Resize((128, 128)),  # Resize to fixed size
    transforms.ToTensor(),           # Convert to tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Normalize
])

# Load dataset using ImageFolder
full_dataset = torchvision.datasets.ImageFolder('petfaces', transform=transform)

print(f"Total images: {len(full_dataset)}")
print(f"Number of classes: {len(full_dataset.classes)}")
print(f"Classes: {full_dataset.classes[:10]}...")  # Show first 10 classes

Now we need to separate dataset into train and test portions:

In [ ]:
# Split dataset into train and test sets (80% train, 20% test)
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size

train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

# Create data loaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

Now let's print the size of tensors in our dataset. If you have done everything correctly, the size of training elements should be
 * `(batch_size,image_size,image_size,3)` for Tensorflow, `batch_size,3,image_size,image_size` for PyTorch
 * `batch_size` for Labels
 
 Labels should contain numbers of classes.

In [ ]:
# Print tensor sizes
data_iter = iter(train_loader)
images, labels = next(data_iter)

print(f"Images tensor shape: {images.shape}")
print(f"Labels tensor shape: {labels.shape}")
print(f"Label values: {labels[:10]}")

In [ ]:
# Display sample images from a batch
def imshow(img, title=None):
    img = img.numpy().transpose((1, 2, 0))
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = std * img + mean  # Unnormalize
    img = np.clip(img, 0, 1)
    plt.imshow(img)
    if title:
        plt.title(title)
    plt.axis('off')

plt.figure(figsize=(15, 6))
for i in range(8):
    plt.subplot(2, 4, i+1)
    imshow(images[i], full_dataset.classes[labels[i]])
plt.tight_layout()
plt.show()

## Define a neural network

For image classification, you should probably define a convolutional neural network with several layers. What to keep an eye for:
* Keep in mind the pyramid architecture, i.e. number of filters should increase as you go deeper
* Do not forget activation functions between layers (ReLU) and Max Pooling
* Final classifier can be with or without hidden layers, but the number of output neurons should be equal to number of classes.

An important thing is to get the activation function on the last layer + loss function right:
* In Tensorflow, you can use `softmax` as the activation, and `sparse_categorical_crossentropy` as loss. The difference between sparse categorical cross-entropy and non-sparse one is that the former expects output as the number of class, and not as one-hot vector.
* In PyTorch, you can have the final layer without activation function, and use `CrossEntropyLoss` loss function. This function applies softmax automatically. 

In [ ]:
# Define a Convolutional Neural Network
class PetCNN(nn.Module):
    def __init__(self, num_classes):
        super(PetCNN, self).__init__()
        
        # Convolutional layers with pyramid architecture
        self.features = nn.Sequential(
            # Block 1: 3 -> 32 filters
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 128 -> 64
            
            # Block 2: 32 -> 64 filters
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 64 -> 32
            
            # Block 3: 64 -> 128 filters
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 32 -> 16
            
            # Block 4: 128 -> 256 filters
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 16 -> 8
        )
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256 * 8 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)  # Flatten
        x = self.classifier(x)
        return x

# Create model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
num_classes = len(full_dataset.classes)
model = PetCNN(num_classes).to(device)

print(f"Model created with {num_classes} classes")
print(f"Device: {device}")
print(model)

## Train the Neural Network

Now we are ready to train the neural network. During training, please collect accuracy on train and test data on each epoch, and then plot the accuracy to see if there is overfitting.

> To speed up training, you need to use GPU if available. While TensorFlow/Keras will automatically use GPU, in PyTorch you need to move both the model and data to GPU during training using `.to()` method in order to take advantage of GPU acceleration. 


In [ ]:
# Training function
def train_epoch(model, dataloader, optimizer, loss_fn, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    
    return total_loss / total, correct / total

def validate(model, dataloader, loss_fn, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            
            total_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    
    return total_loss / total, correct / total

# Train the network
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 15
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print("Starting training...")
for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, loss_fn, device)
    val_loss, val_acc = validate(model, test_loader, loss_fn, device)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"Epoch {epoch+1:2d}/{epochs} | "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

print("Training completed!")

In [ ]:
# Plot training results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot accuracy
ax1.plot(history['train_acc'], 'b-', label='Training Accuracy')
ax1.plot(history['val_acc'], 'r-', label='Validation Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.set_title('Training and Validation Accuracy')
ax1.legend()
ax1.grid(True)

# Plot loss
ax2.plot(history['train_loss'], 'b-', label='Training Loss')
ax2.plot(history['val_loss'], 'r-', label='Validation Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.set_title('Training and Validation Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

# Analysis
best_val_acc = max(history['val_acc'])
best_epoch = history['val_acc'].index(best_val_acc) + 1
print(f"\nBest validation accuracy: {best_val_acc:.4f} at epoch {best_epoch}")

# Check for overfitting
train_val_gap = history['train_acc'][-1] - history['val_acc'][-1]
if train_val_gap > 0.1:
    print(f"⚠️ Potential overfitting detected! Train-Val gap: {train_val_gap:.4f}")
    print("Consider: data augmentation, dropout, early stopping, or more data")
else:
    print(f"Train-Val gap: {train_val_gap:.4f} - No significant overfitting")

What can you say about overfitting? What can be done to improve the accuracy of the model

## Optional: Calculate Top3 Accuracy

In this exercise, we were dealing with classification with quite high number of classes (35), so our result - around 50% validation accuracy - is pretty good. Standard ImageNet dataset has even more - 1000 classes.

In such cases it is difficult to ensure that model **always** correctly predicts the class. There are cases when two breeds are very similar to each other, and the model returns very similar probablities (eg., 0.45 and 0.43). If we measure standard accuracy, it will be considered a wrong case, even though the model did very small mistake. This, we often measure another metric - an accuracy within top 3 most probable predictions of the model.

We consider the case accurate if target label is contained within top 3 model predictions. 

To compute top-3 accuracy on the test dataset, you need to manually go over the dataset, apply the neural network to get the prediction, and then do the calculations. Some hints:

* In Tensorflow, use `tf.nn.in_top_k` function to see if the `predictions` (output of the model) are in top-k (pass `k=3` as parameter), with respect to `targets`. This function returns a tensor of boolean values, which can be converted to `int` using `tf.cast`, and then accumulated using `tf.reduce_sum`.
* In PyTorch, you can use `torch.topk` function to get indices of classes with highers probabilities, and then see if the correct class belongs to them. See [this](https://gist.github.com/weiaicunzai/2a5ae6eac6712c70bde0630f3e76b77b) for more hints.


In [ ]:
# Calculate Top-3 Accuracy
def calculate_topk_accuracy(model, dataloader, device, k=3):
    model.eval()
    correct_topk = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            
            # Get top-k predictions
            _, topk_pred = outputs.topk(k, dim=1)
            
            # Check if true label is in top-k predictions
            correct_topk += topk_pred.eq(labels.view(-1, 1).expand_as(topk_pred)).sum().item()
            total += labels.size(0)
    
    return correct_topk / total

# Calculate top-1 and top-3 accuracy
top1_acc = calculate_topk_accuracy(model, test_loader, device, k=1)
top3_acc = calculate_topk_accuracy(model, test_loader, device, k=3)
top5_acc = calculate_topk_accuracy(model, test_loader, device, k=5)

print(f"Top-1 Accuracy: {top1_acc:.4f} ({top1_acc*100:.2f}%)")
print(f"Top-3 Accuracy: {top3_acc:.4f} ({top3_acc*100:.2f}%)")
print(f"Top-5 Accuracy: {top5_acc:.4f} ({top5_acc*100:.2f}%)")

## Optional: Build Cats vs. Dogs classification

We also want to see how accurate our binary cats vs. dogs classification would be on the same dateset. To do it, we need to adjust labels:

In [ ]:
# Create a binary cats vs dogs dataset
# Cats: breeds starting with lowercase (e.g., 'Abyssinian', 'Bengal', 'Birman' are cats)
# Dogs: breeds starting with uppercase or specific patterns

# Let's check which classes are cats vs dogs
cat_breeds = ['Abyssinian', 'Bengal', 'Birman', 'Bombay', 'British_Shorthair', 
              'Egyptian_Mau', 'Maine_Coon', 'Persian', 'Ragdoll', 'Russian_Blue', 
              'Siamese', 'Sphynx']

print("Classes in dataset:")
for i, cls in enumerate(full_dataset.classes):
    pet_type = "CAT" if cls in cat_breeds else "DOG"
    print(f"  {i}: {cls} ({pet_type})")

In [ ]:
# Create binary labels: 0 = cat, 1 = dog
class BinaryPetDataset(torch.utils.data.Dataset):
    def __init__(self, original_dataset, cat_breeds):
        self.original_dataset = original_dataset
        self.cat_breeds = cat_breeds
        self.classes = original_dataset.classes
        
    def __len__(self):
        return len(self.original_dataset)
    
    def __getitem__(self, idx):
        img, label = self.original_dataset[idx]
        breed_name = self.classes[label]
        # Binary label: 0 for cat, 1 for dog
        binary_label = 0 if breed_name in self.cat_breeds else 1
        return img, binary_label

# Create binary datasets
binary_train_dataset = BinaryPetDataset(train_dataset.dataset, cat_breeds)
binary_test_dataset = BinaryPetDataset(test_dataset.dataset, cat_breeds)

# Create data loaders
binary_train_loader = DataLoader(binary_train_dataset, batch_size=32, shuffle=True)
binary_test_loader = DataLoader(binary_test_dataset, batch_size=32, shuffle=False)

print(f"Binary dataset created!")
print(f"Training samples: {len(binary_train_dataset)}")
print(f"Test samples: {len(binary_test_dataset)}")

# Count cats and dogs
cat_count = sum(1 for _, label in binary_train_dataset if label == 0)
dog_count = len(binary_train_dataset) - cat_count
print(f"Cats: {cat_count}, Dogs: {dog_count}")

# Define binary classifier
class BinaryPetCNN(nn.Module):
    def __init__(self):
        super(BinaryPetCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(128 * 16 * 16, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, 2)  # 2 classes: cat and dog
        )
    
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

# Train binary classifier
binary_model = BinaryPetCNN().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(binary_model.parameters(), lr=0.001)

print("\nTraining binary classifier (Cats vs Dogs)...")
for epoch in range(10):
    train_loss, train_acc = train_epoch(binary_model, binary_train_loader, optimizer, loss_fn, device)
    val_loss, val_acc = validate(binary_model, binary_test_loader, loss_fn, device)
    print(f"Epoch {epoch+1:2d} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

print(f"\nFinal Cats vs Dogs Accuracy: {val_acc:.4f} ({val_acc*100:.2f}%)")